## Text input

https://platform.openai.com/docs/models

In [1]:
from dotenv import load_dotenv
from IPython.display import display, Markdown
load_dotenv()

True

In [ ]:
import boto3
from langchain_aws import ChatBedrock
from langchain_core.messages import SystemMessage, HumanMessage
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.prebuilt import create_react_agent # <--- Importante

llm = ChatBedrock(
    model_id="us.meta.llama4-maverick-17b-instruct-v1:0",  # Nota el prefijo "us."
    region_name="us-east-1",
    model_kwargs={
        "temperature": 0.7,
        "max_tokens": 2048,
        "top_p": 0.9,
    }
)

: 

In [ ]:
from langchain_aws import ChatBedrock
from langchain_core.messages import SystemMessage
from langgraph.graph import StateGraph, MessagesState, END



    

# Initialize the agent with Llama 4
agent = create_agent(
    model=llm,
    system_prompt="You are a science fiction writer, create a capital city at the users request.",
)

ValidationError: 1 validation error for ChatBedrock
model_id
  Input should be a valid string [type=string_type, input_value=ChatBedrock(client=<botoc...ge_tool_message': True}), input_type=ChatBedrock]
    For further information visit https://errors.pydantic.dev/2.12/v/string_type

In [7]:
from langchain.messages import HumanMessage

question = HumanMessage(content=[
    {"type": "text", "text": "What is the capital of The Moon?"}
])

response = agent.invoke(
    {"messages": [question]}
)

display(Markdown(response['messages'][-1].content))

AttributeError: 'NoneType' object has no attribute 'invoke'

## Image input

In [44]:
from ipywidgets import FileUpload
from IPython.display import display

uploader = FileUpload(accept='.png', multiple=False)
display(uploader)

FileUpload(value=(), accept='.png', description='Upload')

In [48]:
print(uploader.value)

({'name': 'mcc.png', 'type': 'image/png', 'size': 3556, 'content': <memory at 0x000002BBF7588280>, 'last_modified': datetime.datetime(2025, 12, 22, 13, 46, 38, 183000, tzinfo=datetime.timezone.utc)},)


In [49]:
import base64

# Get the first (and only) uploaded file dict
uploaded_file = uploader.value[0]

# This is a memoryview
content_mv = uploaded_file["content"]

# Convert memoryview -> bytes
img_bytes = bytes(content_mv)  # or content_mv.tobytes()

# Now base64 encode
img_b64 = base64.b64encode(img_bytes).decode("utf-8")

In [50]:
multimodal_question = HumanMessage(content=[
    {"type": "text", "text": "Tell me about this capital"},
    {"type": "image", "base64": img_b64, "mime_type": "image/png"}
])

response = agent.invoke(
    {"messages": [multimodal_question]}
)

display(Markdown(response['messages'][-1].content))




Welcome to Nova Haven, the capital city of the planet Xylophia-IV, a world renowned for its breathtaking landscapes and advanced technological innovations. Nova Haven is a marvel of urban planning, seamlessly integrating nature with futuristic architecture to create a unique and sustainable environment.

**Overview of Nova Haven**

Nova Haven is situated on the banks of the crystal-clear Lake Xylophia, providing a stunning backdrop to the city's sleek, modern skyline. The city is home to a diverse population of approximately 5 million inhabitants, comprising Xylophians, various alien species, and humans. Nova Haven is a hub of intergalactic commerce, culture, and innovation, attracting visitors and entrepreneurs from across the galaxy.

**Key Features of Nova Haven**

1. **The Aurora Spire**: A majestic, kilometer-high tower that serves as the city's iconic centerpiece. The Aurora Spire is not only a marvel of engineering but also a symbol of the city's commitment to sustainability, as it is powered entirely by renewable energy sources.
   
2. **The Lake Xylophia Promenade**: A picturesque walkway along the lake's edge, offering breathtaking views of the surrounding landscape and the city's skyline. The promenade is lined with parks, cafes, and cultural institutions, making it a popular destination for both locals and tourists.

3. **The Nexus**: Nova Haven's central business district, known for its futuristic skyscrapers and bustling commercial activity. The Nexus is home to the Galactic Stock Exchange, numerous multinational corporations, and a variety of financial institutions.

4. **The University of Nova Haven**: A prestigious academic institution renowned for its research in advanced technologies, environmental science, and intergalactic studies. The university is a hub of innovation, driving technological advancements and fostering collaboration between different species.

5. **The Xylophian Cultural Center**: A celebration of Xylophian heritage and culture, featuring traditional architecture, art, and cuisine. The center hosts various cultural events and festivals throughout the year, promoting understanding and appreciation of Xylophian traditions.

**Infrastructure and Transportation**

Nova Haven boasts a sophisticated transportation network, including the **Nova Haven Hyperloop**, which connects the city to other major urban centers on Xylophia-IV and beyond. The city's public transportation system is efficient and environmentally friendly, utilizing advanced maglev trains and hyper-speed subways.

**Sustainability and Environmental Efforts**

Nova Haven is a leader in sustainable urban development, with a strong focus on renewable energy, green architecture, and eco-friendly

## Audio input

In [51]:
import sounddevice as sd
from scipy.io.wavfile import write
import base64
import io
import time
from tqdm import tqdm

# Recording settings
duration = 5  # seconds
sample_rate = 44100

print("Recording...")
try:
    audio = sd.rec(int(duration * sample_rate), samplerate=sample_rate, channels=1)
    # Progress bar for the duration
    for _ in tqdm(range(duration * 10)):   # update 10x per second
        time.sleep(0.1)
    sd.wait()
    print("Done.")
    
    # Write WAV to an in-memory buffer
    buf = io.BytesIO()
    write(buf, sample_rate, audio)
    wav_bytes = buf.getvalue()
    aud_b64 = base64.b64encode(wav_bytes).decode("utf-8")
    
except Exception as e:
    print(f"Audio recording failed (likely no microphone): {e}")
    print("Using placeholder silent audio for demonstration.")
    aud_b64 = 'UklGRiQAAABXQVZFZm10IBAAAAABAAEAQB8AAIA+AAACAAgAZGF0YQAAAAA='


Recording...
Audio recording failed (likely no microphone): Error querying device -1
Using placeholder silent audio for demonstration.


In [52]:
agent = create_agent(
    model='meta.llama4-maverick-17b-instruct-v1:0',
)

multimodal_question = HumanMessage(content=[
    {"type": "text", "text": "Tell me about this audio file"},
    {"type": "audio", "base64": aud_b64, "mime_type": "audio/wav"}
])

response = agent.invoke(
    {"messages": [multimodal_question]}
)

display(Markdown(response['messages'][-1].content))


ValueError: Unable to infer model provider for model='meta.llama4-maverick-17b-instruct-v1:0', please specify model_provider directly.